# M3_02_HTTP_CLIENT_CHECK

This notebook documents and checks **M3-02: synchronous HttpModelClient**. Run the CPU sections on Colab without a GPU. The optional live section can reuse a local vLLM server or start Qwen3-8B on a Colab A100.

Fill `REPO_URL` with the repository containing these changes. No source files are modified by this notebook. Evidence is written under `/content/m3_02_checks`, outside the checkout. CPU fake-server tests are real HTTP tests, not GPU/model-quality evidence.

Statuses: **PASS** = the stated observed checks passed; **FAIL** = a violation/error was observed; **PARTIAL** = some required evidence is skipped/missing; **NOT VERIFIED** = an optional real-model check was not run. A fake-server PASS never becomes a real-model PASS. Passing client checks does not establish SQL accuracy, held-out PA, determinism or Docker compliance.


## Changes and reasons

| Change | Location | Why |
|---|---|---|
| Synchronous local HTTP client | `src/datamate/llm/http.py` | Replace the HTTP backend's NotImplementedError; reuse the existing ModelClient interface |
| Actual-body hash; explicit decoding, thinking and JSON schema | HTTP request builder | Reproducible requests and future M3-13 replay keys |
| Transport-only schema adapter | `src/datamate/llm/schema.py` | Resolve TimeRef oneOf incompatibility; require distinct explicit kind tags; preserve original contracts |
| Error/retry/timeout handling | HTTP transport | One identical retry on transport failure; no timeout/HTTP/protocol retry |
| Factory and defaulted configuration | `llm/__init__.py`, `config.py` | Select the real client; require an explicit loopback endpoint and matching served model name |
| Additive ModelCall.transport; contracts 1.2.0 | `contracts/types.py`, `contracts/codes.py` | Preserve actual request/response, status, errors and attempts without fabricating text |
| CPU acceptance and updated startup regression | `tests/unit/test_model_client.py`, integration test | Verify the client without weights/GPU; retain the scripted production guard |
| Consumer notice/documentation | README and SPEC §18.5 | Explain new config, hash convention and producer/consumer impact |

The optional transport field defaults to null for older producers. Scripted request hashing is unchanged. Raw generated content/reasoning is never stripped, rewritten, summarized or repaired by the client. Invalid generated JSON/SQL remains the parser/solver's responsibility. A `length` finish reason is preserved.

HTTP is restricted to loopback, ignores proxy environment variables and does not follow redirects. Server launch/health/restarts belong to M4. This notebook is evidence for review, not a statement that Lead/M4 approval or live GPU acceptance already occurred.


## 1. Configuration and repository checkout

Use a clean checkout at a known commit containing M3-02. The notebook does not pull over or reset an existing checkout. Do not put credentials in the Git URL. Live model checks are off by default; they remain NOT VERIFIED until enabled and executed.


In [27]:
from pathlib import Path
from datetime import datetime, timezone
import ast, csv, hashlib, json, os, subprocess, sys, time
import xml.etree.ElementTree as ET

REPO_URL = "https://github.com/Kyoyooo/DataMate.git"  # REQUIRED: your repository URL with the M3-02 changes.
REPO_REF = "origin/m3/M3-02-http-client"  # Recommended: immutable commit.
REPO_DIR = Path("/content/DataMate")
RESULT_DIR = Path("/content/m3_02_checks") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RESULT_DIR.mkdir(parents=True, exist_ok=False)
RUN_REAL_MODEL_CHECK = True
START_LOCAL_VLLM = True  # False: reuse an already running server.
BASE_URL = "http://127.0.0.1:8000/v1"
SERVED_MODEL_NAME = "datamate"
MODEL_PATH = ""  # Optional existing local Qwen3-8B snapshot.
MODEL_REVISION = "b968826d9c46dd6066d109eabc6255188de91218"  # If downloading: REQUIRED immutable commit for Qwen/Qwen3-8B.
VLLM_VERSION = "0.17.0"
SERVER_VENV = Path("/content/m3_02_serving_py312")

def save_json(name, value):
    path = RESULT_DIR / name
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str) + "\n", encoding="utf-8")
    return path

def command(args, **kwargs):
    return subprocess.run([str(x) for x in args], check=True, text=True, **kwargs)

if not REPO_URL.strip():
    raise ValueError("Fill REPO_URL before continuing.")
if not REPO_DIR.exists():
    command(["git", "clone", "--", REPO_URL, REPO_DIR])
elif not (REPO_DIR / ".git").is_dir():
    raise RuntimeError("REPO_DIR exists but is not a Git checkout.")
if REPO_REF:
    command(["git", "-C", REPO_DIR, "checkout", "--detach", REPO_REF])
dirty = subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True).strip()
if dirty:
    raise RuntimeError("Use a clean checkout to identify the exact source under test.")
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
for relative in ["src/datamate/llm/http.py", "src/datamate/llm/schema.py",
                 "tests/unit/test_model_client.py"]:
    assert (REPO_DIR / relative).is_file(), f"Missing M3-02 change: {relative}"
print("Commit:", REPO_COMMIT, "| Evidence:", RESULT_DIR)


Commit: 0c13bca672d9b095f0c657a58e9a94fea2480430 | Evidence: /content/m3_02_checks/20261110T172914_760367Z


## 2. CPU environment and source references

Install the repository CPU lock and editable package. The serving environment is separate and optional. This step requires Internet for setup; tests only use local sockets. A fresh Colab runtime avoids already-imported package versions lingering after installation. The subprocess tests use the installed environment regardless of notebook imports.


In [28]:
command([sys.executable, "-m", "pip", "install", "-r", REPO_DIR / "requirements-cpu.lock"])
command([sys.executable, "-m", "pip", "install", "-e", REPO_DIR, "--no-deps"])
sys.path.insert(0, str(REPO_DIR / "src"))
TEST_ENV = {**os.environ, "PYTHONPATH": str(REPO_DIR / "src")}
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(RESULT_DIR / "cpu_requirements.txt").write_text(freeze, encoding="utf-8")
from IPython.display import display, Markdown
from datamate.config import ModelConfig
from datamate.contracts import CONTRACTS_VERSION, QuestionFrame, SolverOutput, codes
from datamate.llm import (
    HttpModelClient, SCHEMA_ADAPTER_VERSION, http_request_sha256,
    make_client, schema_for_vllm,
)
save_json("environment.json", dict(python=sys.version, repo_commit=REPO_COMMIT,
    contracts_version=CONTRACTS_VERSION, schema_adapter=SCHEMA_ADAPTER_VERSION))
print("Contracts:", CONTRACTS_VERSION, "| Adapter:", SCHEMA_ADAPTER_VERSION)


Contracts: 1.2.0 | Adapter: timeref-kind-required-oneof-to-anyof.v1


## 3. Test runner and evidence

Each criterion below selects actual repository tests and saves stdout/stderr plus JUnit XML. PASS requires at least one test, no failures/errors/skips and exit code zero. Skipped tests produce PARTIAL; test or collection errors produce FAIL. A criterion with only fake-server tests certifies client behavior, not model accuracy.

Repeated notebook runs do not invent prior outputs: this file ships with no executed cells. Use a new RESULT_DIR for a new experiment.


In [29]:
CHECKS = []
def show_rows(rows):
    if not rows:
        display(Markdown("_No rows._"))
        return
    keys = list(rows[0])
    def clean(value):
        return str(value).replace("|", "/").replace("\n", " ")
    text = "| " + " | ".join(keys) + " |\n| " + " | ".join("---" for _ in keys) + " |\n"
    text += "\n".join("| " + " | ".join(clean(row.get(k, "")) for k in keys) + " |" for row in rows)
    display(Markdown(text))

def run_pytest_check(key, title, paths, names=None):
    xml_path = RESULT_DIR / f"{key}.xml"
    args = [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider",
            "--junitxml", str(xml_path), *paths]
    if names:
        args += ["-k", " or ".join(names)]
    result = subprocess.run(args, cwd=REPO_DIR, env=TEST_ENV, text=True,
                            capture_output=True, timeout=300)
    (RESULT_DIR / f"{key}.log").write_text(result.stdout + result.stderr, encoding="utf-8")
    testcases = ET.parse(xml_path).getroot().findall(".//testcase") if xml_path.exists() else []
    failed = sum(t.find("failure") is not None or t.find("error") is not None for t in testcases)
    skipped = sum(t.find("skipped") is not None for t in testcases)
    passed = len(testcases) - failed - skipped
    status = ("FAIL" if result.returncode != 0 or failed or not testcases else
              "PARTIAL" if skipped else "PASS")
    row = dict(criterion=title, status=status, passed=passed, total=len(testcases),
               skipped=skipped, failed=failed, exit_code=result.returncode)
    CHECKS.append(row)
    show_rows([row])
    print(result.stdout[-3000:])
    if result.returncode:
        print(result.stderr[-3000:])
    return row

CLIENT_TEST = ["tests/unit/test_model_client.py"]


## 4. Criterion A — explicit requests, generated-text fidelity, body hashes

The fake server independently inspects every request field: model, messages, decoding, n=1, stream=false, schema, thinking and max_tokens. It returns Vietnamese text/whitespace/reasoning to check exact preservation. Hashes must equal the actual canonical sent bytes, change with effective inputs, and ignore logging-only call_id. Scripted hashing stays unchanged.

PASS means transport fidelity passed. Invalid model-generated JSON/SQL is deliberately preserved; it is not a client transport error. The length finish reason must remain length, not stop.


In [30]:
run_pytest_check("request_fidelity", "Explicit request + exact text + body hash", CLIENT_TEST, [
    "test_http_request_fields_and_exact_response",
    "test_http_hash_covers_actual_body_and_is_order_independent",
    "test_model_text_and_finish_reason_are_not_rewritten",
])


| criterion | status | passed | total | skipped | failed | exit_code |
| --- | --- | --- | --- | --- | --- | --- |
| Explicit request + exact text + body hash | PASS | 6 | 6 | 0 | 0 | 0 |

......                                                                   [100%]



{'criterion': 'Explicit request + exact text + body hash',
 'status': 'PASS',
 'passed': 6,
 'total': 6,
 'skipped': 0,
 'failed': 0,
 'exit_code': 0}

## 5. Criterion B — schema compatibility without relaxing application validation

The adapter must leave the original schema unchanged, require unique kind tags, convert only the known TimeRef union, and reject unknown/overlapping unions. Explicit adapter=none and schema-free requests are tested separately. Preparation failures must return ModelCall.error without sending HTTP. Original Pydantic invariants remain enforced after HTTP.

PASS does not mean every future JSON schema is supported; unknown schemas fail explicitly rather than falling back to unconstrained generation.


In [31]:
run_pytest_check("schema", "Targeted schema adapter + original validation", CLIENT_TEST, [
    "test_schema_adapter_is_targeted_and_does_not_mutate_contract",
    "test_schema_adapter_rejects_unreviewed_unions",
    "test_schema_preparation_error_is_modelcall_not_exception",
    "test_schema_adapter_opt_out_and_schema_free_calls",
    "test_http_parser_still_uses_original_pydantic_validation",
])
original_schema = QuestionFrame.model_json_schema()
wire_schema = schema_for_vllm(original_schema)
save_json("schema_evidence.json", dict(policy=SCHEMA_ADAPTER_VERSION,
    original=original_schema, serving=wire_schema))
print("Original oneOf:", "oneOf" in original_schema["$defs"]["TimeRef"]["properties"]["expr"])
print("Serving anyOf:", "anyOf" in wire_schema["$defs"]["TimeRef"]["properties"]["expr"])


| criterion | status | passed | total | skipped | failed | exit_code |
| --- | --- | --- | --- | --- | --- | --- |
| Targeted schema adapter + original validation | PASS | 7 | 7 | 0 | 0 | 0 |

.......                                                                  [100%]

Original oneOf: True
Serving anyOf: True


## 6. Criterion C — transport retry and timeout

A dropped connection triggers exactly one retry with identical bytes. Two failed attempts return E_MODEL_TRANSPORT. A timeout returns E_MODEL_TIMEOUT without retry. The second transport attempt uses the remaining overall call budget. These are local socket tests with controlled failures.

PASS means the tested policy was observed, not that every possible OS/network failure has been simulated.


In [32]:
run_pytest_check("retry_timeout", "Retry once + shared timeout budget", CLIENT_TEST, [
    "test_transport_retry_once_with_identical_bytes",
    "test_transport_failure_returns_error_after_two_attempts",
    "test_timeout_is_not_retried",
    "test_retry_uses_remaining_call_budget",
])


| criterion | status | passed | total | skipped | failed | exit_code |
| --- | --- | --- | --- | --- | --- | --- |
| Retry once + shared timeout budget | PASS | 4 | 4 | 0 | 0 | 0 |

....                                                                     [100%]



{'criterion': 'Retry once + shared timeout budget',
 'status': 'PASS',
 'passed': 4,
 'total': 4,
 'skipped': 0,
 'failed': 0,
 'exit_code': 0}

## 7. Criterion D — HTTP/protocol failures as values, no external redirects

HTTP 400/429/500/503, malformed JSON and invalid response envelopes must return E_MODEL_ERROR with evidence; they must not silently retry. Redirects must not be followed. HTTP status/body distinguish server rejection from socket failures. Proxy environment variables must not redirect local inference traffic.

PASS means controlled errors were contained in ModelCall.error; it does not imply the server or schema error was fixed.


In [33]:
run_pytest_check("errors", "HTTP/protocol errors + no redirects", CLIENT_TEST, [
    "test_http_error_is_not_retried_and_body_is_retained",
    "test_redirect_is_not_followed",
    "test_invalid_response_protocol_returns_error",
    "test_malformed_json_response_returns_error",
])


| criterion | status | passed | total | skipped | failed | exit_code |
| --- | --- | --- | --- | --- | --- | --- |
| HTTP/protocol errors + no redirects | PASS | 12 | 12 | 0 | 0 | 0 |

............                                                             [100%]



{'criterion': 'HTTP/protocol errors + no redirects',
 'status': 'PASS',
 'passed': 12,
 'total': 12,
 'skipped': 0,
 'failed': 0,
 'exit_code': 0}

## 8. Criterion E — configuration, factory, concurrency and pipeline

The factory must construct HTTP clients in production, reject missing endpoint configuration, and retain the scripted production guard. A shared synchronous client is exercised by concurrent workers. A fake HTTP model drives the real parser → resolver → DuckDB → formatter pipeline and logs actual generated reasoning/SQL.

PASS is integration evidence with a synthetic fixture and fake model. It is not a real-model accuracy or Docker offline result. External server launch remains M4's responsibility.


In [34]:
run_pytest_check("integration", "Factory + concurrent callers + fixture pipeline", CLIENT_TEST, [
    "test_config_restricts_http_to_loopback",
    "test_http_factory_and_scripted_guard",
    "test_shared_client_can_serve_concurrent_callers",
    "test_http_pipeline_executes_real_sql_and_logs_reasoning",
])


| criterion | status | passed | total | skipped | failed | exit_code |
| --- | --- | --- | --- | --- | --- | --- |
| Factory + concurrent callers + fixture pipeline | PASS | 8 | 8 | 0 | 0 | 0 |

........                                                                 [100%]



{'criterion': 'Factory + concurrent callers + fixture pipeline',
 'status': 'PASS',
 'passed': 8,
 'total': 8,
 'skipped': 0,
 'failed': 0,
 'exit_code': 0}

## 9. Full CPU regression and lint

Run all tests after focused checks to detect shared-contract/config regressions. PASS requires all collected tests passing. PARTIAL with skips means the observed CPU checks pass but some optional data-dependent tests were unavailable; inspect the log. Lint PASS means no configured static errors. Neither substitutes for the real Colab call.


In [35]:
run_pytest_check("cpu_regression", "Full CPU regression", ["tests"])
lint = subprocess.run([sys.executable, "-m", "ruff", "check", "src", "tests", "--no-cache"],
                      cwd=REPO_DIR, env=TEST_ENV, text=True, capture_output=True)
(RESULT_DIR / "lint.log").write_text(lint.stdout + lint.stderr, encoding="utf-8")
CHECKS.append(dict(criterion="Ruff", status="PASS" if lint.returncode == 0 else "FAIL",
                  passed=int(lint.returncode == 0), total=1, skipped=0,
                  failed=int(lint.returncode != 0), exit_code=lint.returncode))
show_rows([CHECKS[-1]])
print(lint.stdout, lint.stderr)


| criterion | status | passed | total | skipped | failed | exit_code |
| --- | --- | --- | --- | --- | --- | --- |
| Full CPU regression | PARTIAL | 214 | 216 | 2 | 0 | 0 |

.............................................ss......................... [ 33%]
........................................................................ [ 66%]
........................................................................ [100%]



| criterion | status | passed | total | skipped | failed | exit_code |
| --- | --- | --- | --- | --- | --- | --- |
| Ruff | PASS | 1 | 1 | 0 | 0 | 0 |

All checks passed!
 


## 10. Optional real server: reuse or launch Qwen3-8B

To reuse a server from another notebook/session, set RUN_REAL_MODEL_CHECK=True, START_LOCAL_VLLM=False, BASE_URL and SERVED_MODEL_NAME to its **local** endpoint and served name. Colab sessions do not share localhost: the server must be in this runtime.

For a self-contained A100 check set both flags True. The cell creates an isolated serving environment, obtains a pinned local Qwen3-8B snapshot, applies an approximately 28.8 GiB engine budget, launches one server and waits for health. This is setup support, not a new replica manager. GPU/model downloads happen only when enabled; no real evidence is generated otherwise. Supply MODEL_REVISION before downloading; no automatic latest revision is used.

If setup fails, inspect live_server.log and execute cleanup. Do not mark the real-model check PASS from CPU tests.
Sources: [Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B), [vLLM GPU setup](https://docs.vllm.ai/en/v0.17.0/getting_started/installation/gpu/).


In [36]:
import http.client
from urllib.parse import urlsplit
import signal, shutil
LIVE_SERVER = None
LIVE_SERVER_LOG = None
LIVE_SETUP_ERROR = None
LIVE_ROWS = []

def local_get(base_url, suffix):
    parsed = urlsplit(base_url)
    host = "127.0.0.1" if parsed.hostname == "localhost" else parsed.hostname
    connection = http.client.HTTPConnection(host, parsed.port or 80, timeout=3)
    try:
        connection.request("GET", suffix)
        response = connection.getresponse()
        return response.status, response.read()
    finally:
        connection.close()

if RUN_REAL_MODEL_CHECK and START_LOCAL_VLLM:
    try:
        # Validate loopback URL before any network call.
        ModelConfig(base_url=BASE_URL)
        endpoint = urlsplit(BASE_URL)
        if endpoint.hostname not in {"127.0.0.1", "localhost"}:
            raise ValueError("Self-launch helper binds IPv4 loopback; use an IPv4 BASE_URL.")
        gpu = subprocess.check_output(["nvidia-smi",
            "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"], text=True)
        (RESULT_DIR / "live_gpu.txt").write_text(gpu, encoding="utf-8")
        print(gpu)
        total_mib = float(next(csv.reader(gpu.splitlines()))[1].strip())
        fraction = min(0.90, 28.8 * 1024 / total_mib)
        command([sys.executable, "-m", "pip", "install", "uv==0.8.15", "huggingface-hub==0.34.4"])
        uv = shutil.which("uv")
        if not SERVER_VENV.exists():
            command([uv, "venv", "--python", "3.12", "--seed", SERVER_VENV])
        server_python = SERVER_VENV / "bin/python"
        command([uv, "pip", "install", "--python", server_python, f"vllm=={VLLM_VERSION}"])
        if not MODEL_PATH:
            if not MODEL_REVISION:
                raise ValueError("Set MODEL_REVISION to an immutable Qwen3-8B commit before downloading.")
            from huggingface_hub import snapshot_download
            MODEL_PATH = snapshot_download("Qwen/Qwen3-8B", revision=MODEL_REVISION,
                allow_patterns=["*.safetensors", "*.json", "*.txt", "*.model", "*.jinja", "*.jinja2"])
        if not Path(MODEL_PATH).is_dir():
            raise ValueError("MODEL_PATH must be an existing local snapshot directory.")
        try:
            local_get(BASE_URL, "/health")
        except (OSError, http.client.HTTPException):
            pass
        else:
            raise RuntimeError("Port already occupied; reuse the server or choose a new port.")
        port = endpoint.port or 80
        args = [str(server_python), "-m", "vllm.entrypoints.openai.api_server",
            "--model", str(MODEL_PATH), "--served-model-name", SERVED_MODEL_NAME,
            "--host", "127.0.0.1", "--port", str(port), "--dtype", "bfloat16",
            "--max-model-len", "8192", "--max-num-seqs", "4",
            "--gpu-memory-utilization", str(fraction), "--seed", "20261116",
            "--generation-config", "vllm", "--reasoning-parser", "qwen3"]
        save_json("live_server_command.json", args)
        server_freeze = subprocess.check_output([str(server_python), "-m", "pip", "freeze"], text=True)
        (RESULT_DIR / "live_serving_requirements.txt").write_text(server_freeze, encoding="utf-8")
        LIVE_SERVER_LOG = (RESULT_DIR / "live_server.log").open("w", encoding="utf-8")
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0", "HF_HUB_OFFLINE": "1",
               "TRANSFORMERS_OFFLINE": "1", "VLLM_NO_USAGE_STATS": "1",
               "HF_HUB_DISABLE_TELEMETRY": "1"}
        LIVE_SERVER = subprocess.Popen(args, env=env, stdout=LIVE_SERVER_LOG,
            stderr=subprocess.STDOUT, start_new_session=True)
        started = time.monotonic()
        while True:
            if LIVE_SERVER.poll() is not None:
                raise RuntimeError("vLLM exited. Inspect live_server.log.")
            try:
                status, _ = local_get(BASE_URL, "/health")
                if status == 200:
                    break
            except (OSError, http.client.HTTPException):
                pass
            if time.monotonic() - started > 1200:
                raise TimeoutError("vLLM startup timeout.")
            time.sleep(2)
        print("Live server is healthy.")
    except Exception as exc:
        LIVE_SETUP_ERROR = f"{type(exc).__name__}: {exc}"
        save_json("live_setup_error.json", dict(error=LIVE_SETUP_ERROR))
        print(LIVE_SETUP_ERROR)
elif RUN_REAL_MODEL_CHECK:
    print("Reusing local server:", BASE_URL, "| served name:", SERVED_MODEL_NAME)
else:
    print("Real model setup: NOT VERIFIED (disabled).")


NVIDIA L4, 23034, 580.82.07



Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

... (5 outputs left out)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

Live server is healthy.


## 11. Real-call acceptance through HttpModelClient

The cell uses the production factory (dev=False), repository prompts and schemas, and parse/solve/repair stages. It saves every ModelCall, including actual body/hash, raw response and errors.

Two statuses are separated:
- **Client protocol/fidelity PASS:** HTTP call succeeded, hash matches actual request, generated content/reasoning equal the raw response fields, and backend is http.
- **Schema smoke PASS:** the original Pydantic model accepts complete output (finish_reason=stop). This is not SQL/business correctness.

Disabled check = NOT VERIFIED. Error = FAIL with evidence. One accepted real call is useful M3-02 evidence; all three schema stages are a stronger compatibility smoke test. No full accuracy claim is made.


In [37]:
if RUN_REAL_MODEL_CHECK and LIVE_SETUP_ERROR is None:
    from datamate.catalog import setup_business
    from datamate.contracts import Question
    from datamate.prompts import render_parse, render_solve, render_repair
    from datamate.rules import load_split
    from datamate.temporal import resolve_windows
    from datetime import date

    real_cfg = ModelConfig(base_url=BASE_URL, served_model_name=SERVED_MODEL_NAME,
                           model_call_timeout_s=120)
    real_client = make_client(real_cfg, dev=False)
    split_path = REPO_DIR / "tests/fixtures/split_mini"
    split = load_split(split_path)
    raw = json.loads((split_path / "questions.jsonl").read_text(encoding="utf-8").splitlines()[0])
    q = Question(**{**raw, "as_of_date": date.fromisoformat(raw["as_of_date"]), "line_no": 1})
    cat = setup_business(split_path / "data" / q.business_id, q.business_id, RESULT_DIR / "work")
    scripts = [json.loads(line) for line in
        (REPO_DIR / "tests/fixtures/scripted/split_mini.jsonl").read_text(encoding="utf-8").splitlines()
        if line.strip()]
    gold = QuestionFrame.model_validate_json(next(
        entry["content"] for entry in scripts if entry["stage"] == "parse" and entry["question"] == q.question))
    windows, error = resolve_windows(gold.time, q.as_of_date, split.temporal, cat.data_range)
    if error:
        raise RuntimeError(error)
    solve_messages = render_solve(q, gold, windows, split, cat)
    inputs = [
        ("parse", "parse.frame.v3", render_parse(q, split, cat), QuestionFrame, 1024),
        ("solve", "solve.sql.v1", solve_messages, SolverOutput, 2048),
        ("repair", "repair.sql.v1", render_repair(solve_messages,
            '{"sql":"SELECT deliberately_missing_column FROM ban_hang"}',
            "E_SQL_EXEC", "Referenced column deliberately_missing_column was not found."),
         SolverOutput, 2048),
    ]
    real_calls = []
    for stage, template_id, messages, output_model, max_tokens in inputs:
        call = real_client.complete(stage=stage, template_id=template_id,
            messages=messages, json_schema=output_model.model_json_schema(),
            max_tokens=max_tokens, call_id=f"live:{stage}")
        real_calls.append(call.model_dump(mode="json"))
        fidelity = False
        try:
            message = json.loads(call.transport["response_body"])["choices"][0]["message"]
            fidelity = (call.error is None and call.backend == "http"
                and call.content == message["content"]
                and call.reasoning == message.get("reasoning_content", message.get("reasoning"))
                and call.request_sha256 == http_request_sha256(call.transport["request_body"]))
        except (KeyError, TypeError, ValueError):
            pass
        schema_ok = False
        validation_error = None
        try:
            output_model.model_validate_json(call.content)
            schema_ok = call.error is None and call.finish_reason == "stop"
        except Exception as exc:
            validation_error = str(exc)
        LIVE_ROWS.append(dict(stage=stage, client_fidelity="PASS" if fidelity else "FAIL",
            schema_smoke="PASS" if schema_ok else "FAIL", error=call.error,
            http_status=(call.transport or {}).get("http_status"),
            finish_reason=call.finish_reason, validation_error=validation_error))
    save_json("real_model_calls.json", real_calls)
    save_json("real_model_checks.json", LIVE_ROWS)
    show_rows(LIVE_ROWS)
else:
    LIVE_ROWS = [dict(stage="parse/solve/repair", client_fidelity="NOT VERIFIED",
        schema_smoke="NOT VERIFIED", error=LIVE_SETUP_ERROR or "Real call disabled")]
    show_rows(LIVE_ROWS)


| stage | client_fidelity | schema_smoke | error | http_status | finish_reason | validation_error |
| --- | --- | --- | --- | --- | --- | --- |
| parse | PASS | PASS | None | 200 | stop | None |
| solve | PASS | PASS | None | 200 | stop | None |
| repair | PASS | PASS | None | 200 | stop | None |

## 12. Cleanup and final criterion table

Stop only a vLLM process launched by this notebook; a reused server is left running. The combined conclusion is FAIL if an observed required check failed, otherwise PARTIAL while live evidence or skipped CPU coverage remains. READY FOR REVIEW requires all observed CPU and live smoke criteria, but does not automatically mark the task Done or assert team approval.

Logs can contain runtime prompts, model content and reasoning. Review before sharing. No weights enter the result archive.


In [38]:
if LIVE_SERVER is not None and LIVE_SERVER.poll() is None:
    os.killpg(LIVE_SERVER.pid, signal.SIGTERM)
    try:
        LIVE_SERVER.wait(timeout=30)
    except subprocess.TimeoutExpired:
        os.killpg(LIVE_SERVER.pid, signal.SIGKILL)
        LIVE_SERVER.wait(timeout=10)
if LIVE_SERVER_LOG is not None:
    LIVE_SERVER_LOG.close()
show_rows(CHECKS)
save_json("cpu_checks.json", CHECKS)
live_failed = any(r["client_fidelity"] == "FAIL" or r["schema_smoke"] == "FAIL" for r in LIVE_ROWS)
cpu_failed = any(r["status"] == "FAIL" for r in CHECKS)
fully_observed = (bool(CHECKS) and all(r["status"] == "PASS" for r in CHECKS)
    and len(LIVE_ROWS) == 3 and all(
        r["client_fidelity"] == "PASS" and r["schema_smoke"] == "PASS" for r in LIVE_ROWS))
overall = "FAIL" if cpu_failed or live_failed or LIVE_SETUP_ERROR else (
    "READY FOR REVIEW" if fully_observed else "PARTIAL")
print("M3-02 evidence status:", overall)
save_json("summary.json", dict(overall=overall, cpu=CHECKS, live=LIVE_ROWS,
    repo_commit=REPO_COMMIT, contracts_version=CONTRACTS_VERSION,
    note="No model-quality, held-out PA, Docker or sm_120 acceptance claim."))
with (RESULT_DIR / "criteria.csv").open("w", newline="", encoding="utf-8-sig") as fh:
    writer = csv.DictWriter(fh, fieldnames=list(CHECKS[0]))
    writer.writeheader()
    writer.writerows(CHECKS)
conclusion = """# M3-02 review

- Source commit: {commit}
- Contract version: {version}
- Automated evidence status: {status}
- CPU failed/skipped checks and reasons: [fill]
- Actual server/model/revision and serving stack: [fill]
- Live parse/solve/repair evidence: [fill]
- Runtime error handling and schema compatibility concerns: [fill]
- Shared-interface notice (Lead/M4/M1/M2/replay): [fill]
- Reviewer decision: [fill]
- Follow-up: M3-13 replay; M4 replica/orchestration; parser/solver quality separately.
"""
(RESULT_DIR / "CONCLUSION.md").write_text(
    conclusion.format(commit=REPO_COMMIT, version=CONTRACTS_VERSION, status=overall), encoding="utf-8")
print("Evidence:", RESULT_DIR)


| criterion | status | passed | total | skipped | failed | exit_code |
| --- | --- | --- | --- | --- | --- | --- |
| Explicit request + exact text + body hash | PASS | 6 | 6 | 0 | 0 | 0 |
| Targeted schema adapter + original validation | PASS | 7 | 7 | 0 | 0 | 0 |
| Retry once + shared timeout budget | PASS | 4 | 4 | 0 | 0 | 0 |
| HTTP/protocol errors + no redirects | PASS | 12 | 12 | 0 | 0 | 0 |
| Factory + concurrent callers + fixture pipeline | PASS | 8 | 8 | 0 | 0 | 0 |
| Full CPU regression | PARTIAL | 214 | 216 | 2 | 0 | 0 |
| Ruff | PASS | 1 | 1 | 0 | 0 | 0 |

M3-02 evidence status: PARTIAL
Evidence: /content/m3_02_checks/20261110T172914_760367Z


## 13. Optional download

The archive contains test logs, JUnit results, criteria CSV, schema evidence, source/environment fingerprints, conclusion and optional real ModelCalls. It excludes temporary DuckDB files and model weights. Colab storage is ephemeral.


In [39]:
import zipfile
archive = RESULT_DIR.parent / f"{RESULT_DIR.name}.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as output:
    for path in sorted(RESULT_DIR.rglob("*")):
        if path.is_file() and "work" not in path.relative_to(RESULT_DIR).parts:
            output.write(path, str(path.relative_to(RESULT_DIR)))
print(archive)
try:
    from google.colab import files
except ImportError:
    print("Download through the local file browser.")
else:
    files.download(str(archive))


/content/m3_02_checks/20261110T172914_760367Z.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>